# FlowHMR on Colab: pull up video to 3D motion

Run the cells **in order, top to bottom**: click a cell, then the play button on its left (or Shift+Enter).
Wait for each one to finish (the spinner stops) before starting the next.

Before you start: **Runtime > Change runtime type > T4 GPU > Save**.

You will need: your Hugging Face token, `smplh.tar.xz` and the pull up video. Full guide: `prototype/tools/FLOWHMR.md`.

If anything goes red, copy the error and send it back. If Colab disconnects, start again from cell 1
(downloads are not kept between sessions).

In [ ]:
# 1. Checks that a GPU is attached and sets up the helpers the other cells use.
import os, subprocess, json, glob, shutil, math

REPO = '/content/FlowHMR'            # where FlowHMR gets downloaded
PY = '/content/fenv/bin/python'      # FlowHMR's own Python 3.10 (Colab's Python is a different version)
PIECES_DIR = '/content/pieces'

def run(cmd, cwd=None, env=None, check=True, quiet=False):
    """Run a command, show its output live, return the output text."""
    # Colab tells programs to draw charts inline in the notebook (MPLBACKEND). FlowHMR's own Python
    # doesn't have that and crashes on import, so it gets the plain file backend instead.
    env = dict(os.environ if env is None else env, MPLBACKEND='Agg')
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = []
    for line in p.stdout:
        out.append(line)
        if not quiet: print(line, end='')
    p.wait()
    text = ''.join(out)
    if check and p.returncode != 0:
        if quiet: print(text[-3000:])
        raise RuntimeError(f'Command failed (exit code {p.returncode}). The error is in the lines above.')
    return p.returncode, text

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    raise SystemExit('NO GPU ATTACHED. Go to Runtime > Change runtime type, pick "T4 GPU", click Save, then run this cell again.')
print('GPU found:', r.stdout.strip())

In [ ]:
# 2. Downloads FlowHMR and installs it with its own Python 3.10 (about 5 to 10 minutes, lots of text is normal).
os.chdir('/content')
if not os.path.isdir(REPO):
    run(['git', 'clone', '--depth', '1', 'https://github.com/flowhmr/flowhmr.git', REPO])
# only the three parts video inference needs (skips the physics simulator)
run(['git', 'submodule', 'update', '--init', '--depth', '1',
     'third_party/YOLOX', 'third_party/vggt-omega', 'third_party/sam-3d-body'], cwd=REPO)
print('FlowHMR version (commit):', run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, quiet=True)[1].strip())

# FlowHMR is tested on Python 3.10 + PyTorch 2.5.1 (cu118). Colab ships another Python, so we make a
# separate Python 3.10 with uv and install exactly their versions into it. Colab's own Python is untouched.
run('pip install -q uv', quiet=True)
if not os.path.exists(PY):
    run(['uv', 'venv', '--seed', '--python', '3.10', '/content/fenv'])
run([PY, '-m', 'pip', 'install', '-q', 'wheel', 'ninja'])
run([PY, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'tools/web/requirements.txt'], cwd=REPO)
run([PY, '-m', 'pip', 'install', '-q', '--no-build-isolation', '--no-deps', '-e', 'third_party/YOLOX'], cwd=REPO)

# The T4 has no fast bfloat16. Newer PyTorch says it does (slow emulation), so VGGT would pick it.
# Use float16 on cards older than the A100 generation, like the original VGGT code does.
vg = f'{REPO}/third_party/vggt-omega/vggt_omega/models/vggt_omega.py'
src = open(vg).read()
old = 'torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16'
new = 'torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16'
if old in src:
    open(vg, 'w').write(src.replace(old, new)); print('VGGT set to float16 on this GPU.')
elif new in src:
    print('VGGT float16 fix already applied.')
else:
    print('Note: VGGT code changed upstream, float16 fix not applied (probably fine).')

_, out = run([PY, '-c', 'import torch, yolox; print(torch.__version__, torch.cuda.is_available())'], quiet=True)
print('PyTorch check:', out.strip(), '(expected: 2.5.1+cu118 True)')
if 'True' not in out:
    raise SystemExit('PyTorch cannot see the GPU. Send this output back.')

In [ ]:
# 3. Asks for your Hugging Face token (hidden, never printed) and downloads all model files (several GB, 5+ minutes).
from getpass import getpass
os.chdir(REPO)
tok = getpass('Paste your Hugging Face token, then press Enter: ').strip()
env = dict(os.environ, HF_TOKEN=tok)
HF = '/content/fenv/bin/huggingface-cli'
try:
    os.makedirs('ckpts/yolox', exist_ok=True)
    if not os.path.exists('ckpts/yolox/yolox_l.pth'):
        run(['wget', '-q', '-O', 'ckpts/yolox/yolox_l.pth',
             'https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_l.pth'])
    os.makedirs('assets/body_models', exist_ok=True)
    if not os.path.exists('assets/body_models/smpl_neutral_J_regressor.pt'):
        run(['wget', '-q', '-O', 'assets/body_models/smpl_neutral_J_regressor.pt',
             'https://raw.githubusercontent.com/zju3dv/GVHMR/main/hmr4d/utils/body_model/smpl_neutral_J_regressor.pt'])
    run([HF, 'download', 'facebook/VGGT-Omega', 'vggt_omega_1b_512.pt', '--local-dir', 'ckpts/vggt-omega'], env=env, quiet=True)
    print('VGGT-Omega: ok')
    run([HF, 'download', 'facebook/sam-3d-body-dinov3', '--local-dir', 'ckpts/sam-3d-body-dinov3'], env=env, quiet=True)
    print('SAM 3D Body: ok')
    run([HF, 'download', 'fafsaf1/flowhmr-0.46B', '--include', 'flowhmr_latest/*', 'config.json',
         '--local-dir', 'checkpoints'], env=env, quiet=True)
    print('FlowHMR checkpoint (flowhmr_latest): ok')
except RuntimeError:
    print('\nDownload failed. If the error says 401 or 403: the token is wrong, or your access to '
          'facebook/VGGT-Omega or facebook/sam-3d-body-dinov3 is not approved yet. Run this cell again with a new token.')
    raise
finally:
    del tok, env

In [ ]:
# 4. Puts the SMPL-H body model in place. Pick smplh.tar.xz when the upload button appears (113 MB, takes a minute or two).
import tarfile, numpy as np
from google.colab import files
os.chdir('/content')
dst = f'{REPO}/assets/body_models/smplh/neutral/model.npz'
if not os.path.exists(dst):
    found = glob.glob('/content/smplh*.tar.xz') + glob.glob('/content/model.npz')
    if not found:
        found = [os.path.join('/content', k) for k in files.upload()]
    src = found[0]
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if src.endswith('.tar.xz'):
        with tarfile.open(src) as t:
            m = next((m for m in t.getmembers() if m.name.endswith('neutral/model.npz')), None)
            if m is None:
                raise SystemExit('No neutral/model.npz inside that file. It should be the "Extended SMPL+H model" from mano.is.tue.mpg.de.')
            with t.extractfile(m) as f, open(dst, 'wb') as o: shutil.copyfileobj(f, o)
    else:
        shutil.copy(src, dst)
d = np.load(dst, allow_pickle=True)
print('SMPL-H neutral ok:', d['v_template'].shape[0], 'vertices,', d['J_regressor'].shape[0], 'joints')

In [ ]:
# 5. Startup check: lists any missing model file and makes sure FlowHMR's code loads (models are not loaded yet).
os.chdir(REPO)
need = ['checkpoints/flowhmr_latest/flowhmr_latest.ckpt', 'checkpoints/flowhmr_latest/config.yml',
        'ckpts/yolox/yolox_l.pth', 'ckpts/vggt-omega/vggt_omega_1b_512.pt',
        'ckpts/sam-3d-body-dinov3/model.ckpt', 'ckpts/sam-3d-body-dinov3/model_config.yaml',
        'ckpts/sam-3d-body-dinov3/assets/mhr_model.pt',
        'assets/body_models/smplh/neutral/model.npz', 'assets/body_models/smpl_neutral_J_regressor.pt']
missing = [f for f in need if not os.path.exists(f)]
for f in need: print('ok     ' if f not in missing else 'MISSING', f)
if missing:
    raise SystemExit('Some files are missing: run the cell that downloads them again (3 for models, 4 for SMPL-H).')
# loads every part the real run loads (detector, camera, SAM 3D Body, FlowHMR), without the model weights
run([PY, '-c', 'import sys; sys.path.insert(0, "."); '
     'from flowhmr.pipeline.v2m_inference import V2MInferenceEngine; '
     'import flowhmr.utils.runtime_video_processing, flowhmr.utils.runtime_vggt_camera, flowhmr.utils.runtime_sam_features; '
     'import matplotlib.pyplot; print("FlowHMR code loads: ok")'], cwd=REPO)

In [ ]:
# 6. Upload the pull up video (pulluppullp.mp4) and cut it into one piece per variation.
from google.colab import files
os.chdir('/content')
vids = [v for v in glob.glob('/content/*.mp4')]
if not vids:
    vids = [os.path.join('/content', k) for k in files.upload()]
VIDEO = vids[0]
_, n = run(['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v', '-show_entries',
            'stream=nb_read_frames,width,height', '-of', 'csv=p=0', VIDEO], quiet=True)
print('Video:', os.path.basename(VIDEO), '| width,height,frames =', n.strip())
if not n.strip().endswith(',869'):
    print('WARNING: expected 869 frames. The cut points below were found on the original 29 s TikTok video.')

# Cut points found by looking at the frames (30 fps, first and last frame included).
# Hard cuts are at 8.000 s, 16.000 s and 24.967 s (TikTok outro, thrown away).
PIECES = {
    'chinup_34':    (630, 701),   # 21.0 to 23.4 s: chin up from a 3/4 angle (best for depth). Runs first, it is short.
    'pullup_wide':  (0, 239),     # 0 to 8 s: wide grip pull up, back view
    'pullup_close': (240, 479),   # 8 to 16 s: close grip pull up, back view
    'chinup':       (480, 748),   # 16 to 25 s: chin up, back view then swings to 3/4
}
# The menu bar at the top (three small body icons and the LATS / TRAPS / BICEPS label) is covered with a
# dark block. Otherwise the person tracker follows the highlighted icon (on screen every frame, never
# moving) instead of the mannequin, and FlowHMR turns a still thumbnail into a still body.
MASK = 'drawbox=x=60:y=55:w=455:h=190:color=0x1c2632:t=fill'
os.makedirs(PIECES_DIR, exist_ok=True)
for name, (a, b) in PIECES.items():
    out = f'{PIECES_DIR}/{name}.mp4'
    run(['ffmpeg', '-v', 'error', '-y', '-i', VIDEO, '-vf', f'trim=start_frame={a}:end_frame={b + 1},setpts=PTS-STARTPTS,{MASK}',
         '-an', '-c:v', 'libx264', '-crf', '16', '-pix_fmt', 'yuv420p', '-r', '30', out])
    _, c = run(['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v', '-show_entries',
                'stream=nb_read_frames', '-of', 'csv=p=0', out], quiet=True)
    print(f'{name:13s} frames {a}-{b} -> {c.strip()} frames ({int(c) / 30:.2f} s)')

In [ ]:
# 7. Runs FlowHMR on each piece (roughly 5 to 15 minutes each). If the GPU runs out of memory it retries with lighter settings.
import numpy as np
os.chdir(REPO)
results = {}
for name, (a, b) in PIECES.items():
    frames = b - a + 1
    # Camera frame interval: the camera model looks at every Nth frame. Every frame needs ~30 GB of GPU
    # memory (their measurement), the T4 has 15 GB, so start with about 24 camera frames per piece.
    interval, chunk = max(1, math.ceil(frames / 24)), 8
    while True:
        print(f'\n=== {name}: {frames} frames, camera frame interval {interval}, SAM chunk {chunk} ===')
        rc, log = run([PY, 'run_v2m_demo_generation.py', f'{PIECES_DIR}/{name}.mp4',
                       '--vggt_interval', str(interval), '--sam_chunk_size', str(chunk), '--overwrite'], check=False)
        os.makedirs(f'output/{name}', exist_ok=True)
        with open(f'output/{name}/run_log.txt', 'a') as f: f.write(log)
        ok = rc == 0 and os.path.exists(f'output/{name}/flowhmr_latest.npz')
        oom = 'out of memory' in log.lower() or 'outofmemoryerror' in log.lower()
        if ok:
            # which person did it follow? The mannequin's box is 500+ px tall and below the menu bar.
            bb = np.load(f'output/{name}/{name}_bbox.npz'); box = np.median(bb['bbox'], 0); s, e = bb['start_end']
            h, used = box[3] - box[1], int(e - s + 1)
            person = 'mannequin' if h > 300 and box[3] > 300 else 'WRONG PERSON (too small, probably a menu icon)'
            results[name] = {'ok': True, 'vggt_interval': interval, 'sam_chunk_size': chunk, 'frames': frames,
                             'tracked_frames': [int(s), int(e)], 'box_height_px': int(h), 'person': person}
            print(f'>>> {name}: DONE. Camera frame interval used: {interval}, SAM chunk: {chunk}')
            print(f'>>> followed: {person}, box {int(h)} px tall, frames {s} to {e} ({used} of {frames})')
            break
        if oom and interval < 30:
            interval, chunk = min(30, interval * 2), max(4, chunk // 2)
            print(f'>>> {name}: ran out of GPU memory, retrying with camera frame interval {interval}, SAM chunk {chunk}')
            continue
        results[name] = {'ok': False, 'vggt_interval': interval, 'sam_chunk_size': chunk, 'out_of_memory': oom}
        print(f'>>> {name}: FAILED' + (' (out of memory even at the lightest settings)' if oom else
              '. Not a memory problem: copy the red error above and send it back.'))
        break
json.dump(results, open('output/run_settings.json', 'w'), indent=1)
print('\nSummary:'); [print(f'  {k:13s}', v) for k, v in results.items()]

In [ ]:
# 8. Shows what came out and downloads everything in one zip (flowhmr_results.zip).
import numpy as np, zipfile
from google.colab import files
os.chdir(REPO)
zp = '/content/flowhmr_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('output/run_settings.json', 'run_settings.json')
    for name in PIECES:
        d = f'output/{name}'
        if not os.path.isdir(d): continue
        res = f'{d}/flowhmr_latest.npz'
        if os.path.exists(res):
            m = np.load(res)
            print(f"{name:13s} poses {m['poses'].shape}  trans {m['trans'].shape}  "
                  f"height range of the hips {m['trans'][:, 1].min():.2f} to {m['trans'][:, 1].max():.2f} m")
        else:
            print(f'{name:13s} no result')
        # the motion, the camera and person boxes (small), and the log; not the big feature caches
        for f in glob.glob(f'{d}/*.npz') + glob.glob(f'{d}/*.txt'):
            z.write(f, f'{name}/{os.path.basename(f)}')
print('\nZip size:', round(os.path.getsize(zp) / 1e6, 1), 'MB')
files.download(zp)
print('If no download started: open the folder icon on the left, find flowhmr_results.zip in /content, right click > Download.')